# 01 — PDF Extraction & Cleaning

This notebook extracts proportional-candidate rows from the Election Commission PDF and saves a clean CSV for analysis.

Variables of interest:
- **लिङ्ग** (Gender)
- **समावेशी समूह** (Inclusive Group)
- **नागरिकता जारी जिल्ला** (Citizenship-issuing district)

> The PDF mixes Nepali + English. During extraction, some Nepali diacritics can be lost; we normalize the main categories robustly.

In [ ]:
# Paths
import sys
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
# Add project root to sys.path so `src` package can be imported
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PDF_PATH = PROJECT_DIR / "data" / "raw" / "PreliminaryCandidateListPR_2082_10_06.pdf"
OUT_CSV = PROJECT_DIR / "data" / "processed" / "candidates.csv"

FIG_DIR = PROJECT_DIR / "outputs" / "figures"
TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH

In [ ]:
import pandas as pd
from src.utils import extract_candidates_from_pdf
import importlib
import src.canonical as canonical
importlib.reload(canonical)

standardize_party_district = canonical.standardize_party_district
standardize_inclusive_group = canonical.standardize_inclusive_group
CANONICAL_PARTIES = canonical.CANONICAL_PARTIES
CANONICAL_DISTRICTS = canonical.CANONICAL_DISTRICTS
CANONICAL_INCLUSIVE_GROUPS = canonical.CANONICAL_INCLUSIVE_GROUPS

df = extract_candidates_from_pdf(str(PDF_PATH))
df = standardize_party_district(df)
df = standardize_inclusive_group(df)

# Save parsing mismatches for manual review
party_unknown = (df.loc[df['party'] == 'UNKNOWN_PARTY', ['party_raw']]
                 .value_counts()
                 .rename('count')
                 .reset_index()
                 .sort_values('count', ascending=False))
district_unknown = (df.loc[df['नागरिकता जारी जिल्ला'] == 'UNKNOWN_DISTRICT', ['district_raw']]
                    .value_counts()
                    .rename('count')
                    .reset_index()
                    .sort_values('count', ascending=False))
group_unknown = (df.loc[df['समावेशी समूह'] == 'UNKNOWN_GROUP', ['group_raw']]
                 .value_counts()
                 .rename('count')
                 .reset_index()
                 .sort_values('count', ascending=False))
party_unknown.to_csv(TABLE_DIR / 'party_unknown_values.csv', index=False, encoding='utf-8')
district_unknown.to_csv(TABLE_DIR / 'district_unknown_values.csv', index=False, encoding='utf-8')
group_unknown.to_csv(TABLE_DIR / 'group_unknown_values.csv', index=False, encoding='utf-8')

df.head()


In [ ]:
# Quick sanity checks
print('Rows:', len(df))
print('Unique parties:', df['party'].nunique())
print('Unique districts:', df['नागरिकता जारी जिल्ला'].nunique())
print('Unique inclusive groups:', df['समावेशी समूह'].nunique())
print('Unknown parties:', int((df['party'] == 'UNKNOWN_PARTY').sum()))
print('Unknown districts:', int((df['नागरिकता जारी जिल्ला'] == 'UNKNOWN_DISTRICT').sum()))
print('Unknown groups:', int((df['समावेशी समूह'] == 'UNKNOWN_GROUP').sum()))
print('Gender values:', df['लिङ्ग'].value_counts(dropna=False).to_dict())
print('Inclusive groups:', df['समावेशी समूह'].value_counts(dropna=False).to_dict())

# Guardrails against drift from the provided canonical lists
party_outside = sorted(set(df['party'].unique()) - set(CANONICAL_PARTIES) - {'UNKNOWN_PARTY'})
district_outside = sorted(set(df['नागरिकता जारी जिल्ला'].unique()) - set(CANONICAL_DISTRICTS) - {'UNKNOWN_DISTRICT'})
group_outside = sorted(set(df['समावेशी समूह'].unique()) - set(CANONICAL_INCLUSIVE_GROUPS) - {'UNKNOWN_GROUP'})
print('Outside-party values:', party_outside)
print('Outside-district values:', district_outside)
print('Outside-group values:', group_outside)


In [ ]:

# Save processed CSV
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
OUT_CSV


In [ ]:
# Export a basic overview table (canonical parties only)
party_counts = (df[df['party'] != 'UNKNOWN_PARTY']
                .groupby('party', as_index=False)
                .size()
                .rename(columns={'size':'candidates'})
                .sort_values('candidates', ascending=False))
party_counts.to_csv(TABLE_DIR / 'party_candidate_counts.csv', index=False, encoding='utf-8')
party_counts.head(20)


In [ ]:
# Optional: quick plot of candidate counts per party (top 20)
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Kohinoor Devanagari'

top = party_counts.head(20).copy()
plt.figure(figsize=(10, 6))
plt.barh(top["party"][::-1], top["candidates"][::-1])
plt.xlabel("Candidates")
plt.title("Top 20 parties by number of PR candidates")
plt.tight_layout()
out = FIG_DIR / "top20_parties_by_candidates.png"
plt.savefig(out, dpi=200)
plt.show()
out